In [234]:
import manim as mn
import random
from manim import *

config.media_width = "75%"
config.verbosity = "WARNING"

print(mn.__version__)

0.20.1


In [318]:
%%manim -qm ExamplePoint

# https://docs.manim.community/en/stable/reference/manim.utils.color.manim_colors.html

#####################################
##################################### Wetlab protocol
#####################################

def create_bactera():
    a = [-1,-1, 0]
    b = [ 1,-1, 0]
    c = [ 1, 1, 0]
    d = [-1, 1, 0]
    bact_inner_color = GRAY
    ap1 = ArcPolygon(a, b, c, d, fill_opacity=1, color=GRAY, arc_config=[
        {'radius': 10, 'color': bact_inner_color}, 
        {'radius': 1, 'color': bact_inner_color},
        {'radius': 10, 'color': bact_inner_color},
        {'radius': 1, 'color': bact_inner_color}])
    return(ap1)

def create_bact_genome():
    bscale = 0.8
    rng = np.random.default_rng(seed=1)
    list_point = [[
        rng.uniform(-bscale, bscale),
        rng.uniform(-bscale, bscale),
        0] for i in range(15) ]
    ap1 = ArcPolygon(*list_point, color=YELLOW, angle=2)
    return(ap1)
    
def create_spc():
    spc_radius = 2
    blue_circle = Circle(fill_color=BLUE, stroke_color=BLACK, fill_opacity=0.5, radius=spc_radius)
    spc_outer = DashedVMobject(Circle(radius=spc_radius, color=WHITE), dashed_ratio=0.5, num_dashes=30)
    #return Group(blue_circle, spc_outer)
    return [blue_circle, spc_outer]

def create_lysis_particles():
    colorList = [RED, GREEN, BLUE, YELLOW]
    points = [Point(location=[np.random.uniform(-8, 8),  np.random.uniform(-4, 4), 0], color=np.random.choice(colorList))
              for i in range(1000)]    
    return Group(*points)

class ExamplePoint(Scene):
    def construct(self):
        bact_env = create_bactera()
        bact_genome = create_bact_genome()
        bact = Group(bact_env,bact_genome).scale(0.5)
        spc = create_spc()

        lysis_part = create_lysis_particles()
        lysis_part.shift([-20,0,0])
        
        self.add(lysis_part)
        self.add(spc[0])
        self.add(spc[1])
        self.add(bact)

        self.play(
            AnimationGroup(
                Create(spc[0],run_time=3),
                Create(spc[1],run_time=3),
            )
        )   
        
        self.play(bact.animate().shift([-1, 0,0]))
        self.play(bact.animate().shift([ 1, 1.2,0]))
        self.play(bact.animate().shift([ 1,-1.2,0]))
        self.play(bact.animate().shift([-1, 0,0]))

        self.play(lysis_part.animate(run_time=5).shift([20,0,0]))        
        self.play(bact_env.animate(run_time=3).set_opacity(0))
        self.play(lysis_part.animate(run_time=3).shift([20,0,0]))

        self.wait()
        

Manim Community v0.20.1

In [405]:
%%manim -qm MobjectMatrixExample

#####################################
##################################### Can turn DNA into a sequence ID
#####################################

class MobjectMatrixExample(Scene):
    def construct(self):
        m0 = MobjectMatrix([
            [Text("A"), Text("00")], 
            [Text("T"), Text("01")],
            [Text("C"), Text("10")],
            [Text("G"), Text("11")]
        ])
        m0.move_to([-4,0,0])

        txt_head1 = Tex(r"We can assign a number \\ to each DNA sequence!")
        txt_head1.move_to([2,2,0])
        self.add(txt_head1)
        
        dna_txt1 = Text(" A")
        dna_txt2 = Text(" A")
        dna_txt3 = Text(" T")
        dna_txt4 = Text(" C")
        grp_a =  VGroup(dna_txt1, dna_txt2, dna_txt3, dna_txt4).arrange()
        grp_a.next_to(txt_head1, DOWN)
        #grp_a.move_to(RIGHT)

        
        txt1 = Text("00")
        txt2 = Text("00")
        txt3 = Text("01")
        txt4 = Text("11")
        grp_b =  VGroup(txt1, txt2, txt3, txt4).arrange()
        grp_b.next_to(grp_a, DOWN)
        
        txt_dec = Text("Decimal: 7") #00000111 1 + 2 + 4 = 7
        txt_dec.next_to(grp_b, DOWN)

        ####### just DNA ##########
        self.wait()
        self.play(
            Create(dna_txt1),
            Create(dna_txt2),
            Create(dna_txt3),
            Create(dna_txt4)
        )
        self.wait()
        self.add(m0)
        self.add(grp_a)
        self.wait()

        ####### Binary conversion ##########
        self.play(Create(txt1))
        self.play(Create(txt2))
        self.play(Create(txt3))
        self.play(Create(txt4))
        self.wait()

        ####### Decimal conversion ##########
        self.play(Create(txt_dec))
        self.wait()



Manim Community v0.20.1

In [ ]:
%%manim -qm InputToGraphPointExample

#        cmatrix = Matrix([[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]],
#            element_alignment_corner=UL,
#            left_bracket="(",
#            right_bracket=")").scale(0.4).move_to([0,-2,0])
#        self.add(cmatrix)

def get_wx_dna(x):
    return(x*scale_dna_x - 5 )

def get_wx_kmer(x):
    return(x*scale_kmer_x - 5 )

class InputToGraphPointExample(Scene):
    def construct(self):

        #######
        txt_head1 = Tex(r"Using number for the DNA KMER,\\ we can fill in a vector of KMER counts")
        txt_head1.move_to([0,2,0])
        self.add(txt_head1)
        self.wait()


        ######### constants #################
        scale_dna_x = 0.3
        scale_kmer_x = 0.25
        window_size = scale_dna_x*5
        kmer_size = 5
        kmer_flanks = int(kmer_size/2)
        height_kmer_rect = 0.7
        pos_kmer_y1 = -height_kmer_rect/2.0
        pos_kmer_y = -2

        dna_seq = "AACGACAGTACCGA"#CATCGACACTACTACG"
        #into_pos = list(range(len(dna_seq)))
        into_pos = list(range(30))
        random.shuffle(into_pos)
        
        ######### Show count vector #################
        

        ## Place DNA
        for i in range(len(dna_seq)):
            text_dna = Text(dna_seq[i], font_size=20).move_to(([get_wx_dna(i), 0, 0]))
            self.add(text_dna)

        self.wait()
        
        ## Place matrix
        num_mat = [0 for i in range(len(into_pos))]
        text_mat = [Text("0", font_size=15).move_to([get_wx_kmer(i), pos_kmer_y, 0]) for i in range(len(num_mat))]
        for t in text_mat:
            self.add(t)
            
        self.wait()
        
        ######### animate the counting #################

        ## Prep rect                 
        pos_dna_rect = [get_wx_dna(0), 0, 0]
        rect_kmer = Rectangle(width=window_size, height=height_kmer_rect).move_to(pos_dna_rect)
        self.add(rect_kmer)
        self.play(Create(rect_kmer))
        
        ## Prep arrow  
        pdna_arrow = [get_wx_dna(0), pos_kmer_y1, 0]
        pmat = [get_wx_kmer(into_pos[0]), pos_kmer_y, 0]            
        arrow_to_mat = Arrow(pdna_arrow, pmat, buff=0, color=RED)  
        self.add(arrow_to_mat)
        
        pos_dna_rect = [get_wx_dna(0), 0, 0]
        pmat = [get_wx_kmer(into_pos[0]), pos_kmer_y, 0]
        rect_kmer.move_to(pos_dna_rect)

        for i in range(kmer_flanks,-kmer_flanks+len(dna_seq)):
            pos_dna_rect = [get_wx_dna(i), 0, 0]
            pdna = [get_wx_dna(i), pos_kmer_y1, 0]
            mat_index = into_pos[i]
            num_mat[mat_index] = num_mat[mat_index] + 1
            pmat = [get_wx_kmer(mat_index), pos_kmer_y, 0]

            new_text = Text(str(num_mat[mat_index]), font_size=20).move_to(([get_wx_kmer(mat_index), pos_kmer_y, 0]))
            new_arrow_to_mat = Arrow(pdna, pmat, buff=0, color=RED)  #[0,0,0]

            self.play(
                rect_kmer.animate.move_to(pos_dna_rect),
                arrow_to_mat.animate.become(new_arrow_to_mat)#set_points_by_ends(pdna, pmat),
            )
            self.play(
                text_mat[mat_index].animate.become( new_text ),
            )
        
        self.wait()

In [ ]:
%%manim -qm ExamplePoint


#####################################
##################################### Countsketch: need to use angles for comparison
#####################################

do_full_anim = True

def make_samples_for_genome_vec(x, rate_lambda):
    return([np.sign(y)*np.random.poisson(np.abs(y)*rate_lambda) for y in x ])
    
def rescale_gvec(x):
    scale_gvec = 1/100.0
    s = [y*scale_gvec for y in x]
    return(s)

class ExamplePoint(ThreeDScene):
    def construct(self):        
        axes = ThreeDAxes()
        self.set_camera_orientation(phi=20 * DEGREES, theta=-70 * DEGREES)
        self.add(axes)

        font_size_strain=30
        font_size_math=30

        color_samples = BLUE
        color_genomes = RED

        self.begin_ambient_camera_rotation(1*DEGREES, about='phi')

        ####### Text up left #############
        text_orig_c = Tex(r"Genome KMER counts: $C_i$",font_size=font_size_math, color=color_genomes)
        text_samp_c = Tex(r"Sequenced counts: $\tilde{C_i} \sim Poi(\lambda C_i) $; $\lambda$ is coverage", font_size=font_size_math, color=color_samples)
        text_samp_c.next_to(text_orig_c, DOWN)
        grp_ul = VGroup(text_orig_c, text_samp_c)
        self.add_fixed_in_frame_mobjects(grp_ul)
        grp_ul.to_corner(UL)
        text_orig_c.set_opacity(0)
        text_samp_c.set_opacity(0)
        
        ####### Genome vectors #############
        coord_len = 100.0
        coord_cell1 = [-coord_len, coord_len, 0.0]
        arrow_cell1 = Arrow(ORIGIN, rescale_gvec(coord_cell1), buff=0, color=color_genomes)
        text_cell1 = Tex('$C_1$: Bacillus', font_size=font_size_strain).next_to(arrow_cell1.get_end(), RIGHT)

        coord_len = 150.0
        coord_cell2 = [-coord_len, -coord_len, 0.0]
        arrow_cell2 = Arrow(ORIGIN, rescale_gvec(coord_cell2), buff=0, color=color_genomes)
        text_cell2 = Tex('$C_2$: Campylobacter', font_size=font_size_strain).next_to(arrow_cell2.get_end(), RIGHT)
        
        if do_full_anim:
            text_orig_c.set_opacity(1)
            self.play(Create(arrow_cell1))
            self.play(Create(text_cell1))
            self.play(Create(arrow_cell2))
            self.play(Create(text_cell2))
        else:            
            self.add(text_orig_c)
            self.add(arrow_cell1, text_cell1)
            self.add(arrow_cell2, text_cell2)
            
        if do_full_anim:
            self.wait()

        ####### Cells as samples #############    
        def add_samples(c):        
            #list_lambda = [ (i*2.0)/10.0 for i in range(10)]        
            list_lambda = [ (i*1.5)/10.0 for i in range(10)]        
            list_s = [make_samples_for_genome_vec(c, r) for r in list_lambda]
            #list_arrow = [Arrow(ORIGIN, rescale_gvec(s), buff=0, color=BLUE, stroke_width=1) for s in list_s]
            list_arrow = [Line(ORIGIN, rescale_gvec(s), buff=0, color=color_samples, stroke_width=3) for s in list_s]
            return(list_arrow)
        
        list_arrow1 = add_samples(coord_cell1)
        list_arrow2 = add_samples(coord_cell2)
        comb_list = list_arrow1+list_arrow2
        random.shuffle(comb_list)        
        
        if do_full_anim:
            self.play(text_samp_c.animate().set_opacity(1))
            for a in comb_list:
                self.play(Create(a, run_time=0.1))
        else:
            text_samp_c.set_opacity(1)
            for a in comb_list:
                self.add(a)

        if do_full_anim:
            self.wait()

        ####### Euclidian distance cannot be used #############
        p1 = list_arrow1[9].get_end()
        p2 = list_arrow2[9].get_end()
        b1 = BraceBetweenPoints(p1,p2)
        b1text = b1.get_text("Dissimilarity").scale(0.8)

        p1 = list_arrow1[2].get_end()
        p2 = list_arrow2[3].get_end()
        b2 = BraceBetweenPoints(p1,p2)
        b2text = b1.get_text("??")
        
        if do_full_anim:
            self.play(Create(b1), Create(b1text))
            self.wait()
            self.play(Create(b2), Create(b2text))
        else:
            self.add(b1, b1text)
            self.add(b2)
            
        if do_full_anim:
            self.wait()

        
        ####### Use angle instead #############
        self.remove(b1)
        self.remove(b1text)
        self.remove(b2)
        self.remove(b2text)
        
        # Create the angle arc
        angle_genomes = Angle(arrow_cell1, arrow_cell2, radius=0.5, color=YELLOW)
        angle_label = Tex(r"Dissimilarity",font_size=font_size_math, color=YELLOW).next_to(angle_genomes, [-2,-1,0], buff=0.1)

        if do_full_anim:
            self.play(Create(angle_genomes), Create(angle_label))
        else:
            self.add(angle_genomes, angle_label)        
            
        if do_full_anim:
            self.wait()
        
        self.stop_ambient_camera_rotation(about='phi')
        

Manim Community v0.20.1

Animation 9: Create(Line):   0%|                                                                                                                                                                                                                                                   | 0/3 [00:00<?, ?it/s]

In [ ]:
%%manim -qm ExamplePoint

#####################################
##################################### Countsketch: Angle is a dot product
#####################################

do_full_anim = True

class ExamplePoint(ThreeDScene):
    def construct(self):        
        foo()





In [ ]:
%%manim -qm ExamplePoint

#####################################
##################################### Countsketch: Jaccard index is a dot product
#####################################

do_full_anim = True

class ExamplePoint(ThreeDScene):
    def construct(self):        
        foo()




In [ ]:

%%manim -qm ExamplePoint

#####################################
##################################### Countsketch: There are too many dimensions
#####################################

do_full_anim = True

class ExamplePoint(ThreeDScene):
    def construct(self):        
        foo()





In [ ]:


%%manim -qm ExamplePoint

#####################################
##################################### Countsketch: Random projection
#####################################

do_full_anim = True

class ExamplePoint(ThreeDScene):
    def construct(self):        
        foo()



